In [1]:
# 01 · STATE STAGE ACT 단일 정책 CONFIG · GitHub 중단 복구 · Drive 미사용
CFG = {
    # 저장 · ver2는 기존 DP 결과와 별도 Release에 저장
    'run_name': 'moveboxes_stage_anchor_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': '4730e696be227c3c50d80f1c96af879897bf7511',
    'output_root': '/content/moveboxes_runs',

    # 데이터 / Colab 2026.07 · Python 3.12 · T4
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 학습 · 시뮬레이터 없이 CPU 데이터 + GPU 모델
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'total_iters': {'easy': 12000, 'medium': 20000, 'hard': 30000},
    'amp': True,

    # 작은 State ACT · 기존 DP 체크포인트 사용 불가
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,

    # 검증 / 중단 복구 / 작은 관측 위치 증강
    'save_freq': 1000,
    'warmup_steps': 500,
    'validation_batches': 8,
    'kl_weight': 0.001,
    'position_noise': 0.001,

    # 실행 · 매 스텝 재계획, 최근 XYZ 예측 평균, 집게는 최신 예측
    'temporal_decay': 0.25,
    'ensemble_window': 4,
    'ensemble_candidates': [1, 4],

    # 빠른 테스트 / 최종 평가 · 시드 분리, 기존 200스텝 유지
    'test_episodes': 8,
    'test_seed_start': 40000,
    'test_record_video': True,
    'tuning_episodes': 8,
    'tuning_seed_start': 20000,
    'benchmark_episodes': 100,
    'eval_seed_start': 30000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    'record_eval_video': True,

    # 출력
    'console_interval_seconds': 10,
    'team': 'my-team',

    # 실행 조건으로 행동 학습 · 빠른 테스트가 0이면 긴 평가 생략
    'action_training_mode': 'prior',
    'repair_iters': 2000,
    'allow_zero_success_evaluation': False,

    # 단계 판단 · 학습된 완료/복구 확신이 낮으면 현재 단계 유지
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'stage_loss_weight': 0.3,
    'gate_loss_weight': 0.3,

    # 복구 시연 · 수집 전용 expert, 학습/제출은 학습된 정책
    'recovery_episodes': 16,
    'recovery_max_attempts': 48,
    'recovery_seed_start': 100000,
    'collection_max_steps': {'easy': 500, 'medium': 900, 'hard': 1400},
    'noise_probability': 0.08,
    'action_noise_std': 0.12,
    'drop_probability': 0.015,

}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','stage_chunk_policy',
             'stage_pick_sampling','stage_pick_train','stage_all_pick_retrain',
             'stage_pick_finetune','stage_pick_diagnose','stage_reference_check',
             'stage_anchor_continue','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from stage_experiment import StageExperiment, source_bundle
experiment = StageExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [3]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


GitHub 토큰 입력 (이 런타임에서만 사용): ··········
GitHub 복원: 0 files
로컬 작업 경로: /content/moveboxes_runs/moveboxes_stage_anchor_v1_benchmark
실행 노트북 사본: /content/moveboxes_runs/moveboxes_stage_anchor_v1_benchmark/moveboxes_stages_colab.ipynb
새 런타임에서는 01~03 셀로 결과를 복원하고, 학습·평가는 04~05 셀 준비 후 실행합니다.
GitHub 백업 완료: common (52 files)
단계 ACT: 학습한 완료/복구 판단 + 집기/운반/놓기 행동, 별도 실험
easy   |      -- | 모델 없음 | 학습완료 기록 False
medium |      -- | 모델 없음 | 학습완료 기록 False
hard   |      -- | 모델 없음 | 학습완료 기록 False
가중 점수: 0.0000 / 미평가: ['easy', 'medium', 'hard']


In [4]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


실행: nvidia-smi
전체 로그: /content/moveboxes_runs/moveboxes_stage_anchor_v1_benchmark/sessions/20260915_064732_a2f260/gpu.log
완료
실행: git clone https://github.com/marso-robotics/berlin-marso-hackathon.git /content/berlin-marso-hackathon
전체 로그: /content/moveboxes_runs/moveboxes_stage_anchor_v1_benchmark/sessions/20260915_064732_a2f260/commands.log
완료
실행: /usr/bin/python3 -m pip install mani-skill==3.0.1 sapien==3.0.3 diffusers==0.38.0 hydra-core omegaconf gymnasium tyro h5py kagglehub tensorboard matplotlib transforms3d imageio[ffmpeg]
전체 로그: /content/moveboxes_runs/moveboxes_stage_anchor_v1_benchmark/sessions/20260915_064732_a2f260/install.log
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 941.9/941.9 kB 72.2 MB/s eta 0:00:00
Successfully uninstalled diffusers-0.39.0
완료
실행: /usr/bin/python3 -m pip install -e /content/berlin-marso-hackathon
전체 로그: /content/moveboxes_runs/moveboxes_stage_anchor_v1_benchmark/sessions/20260915_064732_a2f260/install_repo.log
완료
설치 완료. 다음 데이터·GPU 확인 셀을 실행하세요.


In [5]:
# 05 · GitHub 데이터 다운로드·검증 / GPU와 정책 실행 확인
experiment.prepare_data()
experiment.check_runtime()


GitHub 데이터 다운로드: state (65.4 MiB)
easy 200 demos
medium 200 demos
hard 200 demos
GitHub 백업 완료: common (58 files)
실행: /usr/bin/python3 /content/berlin-marso-hackathon/test_policy_runtime.py
전체 로그: /content/moveboxes_runs/moveboxes_stage_anchor_v1_benchmark/sessions/20260915_064732_a2f260/policy_check.log
Stage ACT 54: backward/load/action/reset OK (cuda)
완료
실행: /usr/bin/python3 -c import torch
from warehouse_sort.utils import compose_cfg, make_env
assert torch.cuda.is_available(), 'GPU runtime required'
for level, dims in [('easy',54),('medium',72),('hard',90)]:
    cfg = compose_cfg(['difficulty='+level, 'num_envs=1'])
    env, _ = make_env(cfg, 'state', cfg.randomization, num_envs=1, render_mode='rgb_array')
    try:
        obs, _ = env.reset(seed=42)
        assert tuple(obs.shape)==(1,dims)
        env.step(torch.zeros((1,4),device='cuda'))
        assert env.render() is not None
        print(level, 'GPU/render OK')
    finally:
        env.close()

전체 로그: /content/moveboxes_runs/

# 난이도별 검증 Stage ACT 복원 및 독립 추가 학습

Easy 100% `block_03.pt`, Medium 40.625% `block_02.pt`, Hard 4.167% `best_val.pt`를 SHA-256으로 확인해 각각 복원합니다. Easy는 고정하며 Medium과 Hard는 서로 다른 초기값·optimizer·결과 폴더로 학습합니다. 한 난이도의 학습은 다른 난이도 가중치를 변경하지 않습니다.

01~05는 기존 GitHub 토큰, 데이터 다운로드, T4 설치 과정과 같습니다. Drive는 사용하지 않습니다. 06은 세 앵커와 각 난이도의 기존 recovery 시연을 복원합니다. 07은 학습 전 공식 테스트입니다. 09와 11은 Medium/Hard 추가 학습입니다. 500 step마다 optimizer·AMP scaler·RNG까지 GitHub Release에 저장합니다.

추가 학습 모델은 원본 앵커를 덮어쓰지 않습니다. 마지막 셀에서 공식 결과를 확인한 뒤 `USE_TRAINED`를 직접 지정합니다. 기본값은 모두 `False`라서 검증 앵커가 유지됩니다.


In [6]:
# 06 · 세 앵커 + 기존 recovery 시연 복원 / baseline candidate 생성
import json, os, shutil, signal, subprocess, sys
from pathlib import Path
from IPython.display import Video, display
from stage_anchor_continue import ANCHORS, prepare, train, package

CONTINUE_ITERS = {'medium':8000, 'hard':12000}
CONTINUE_LR = 2e-5
MAX_STEPS = 200
UPSTREAM = Path(CFG['repo_dir'])
OFFICIAL = UPSTREAM/'conf/eval/default.yaml'
anchor_exp = prepare(experiment, iterations=CONTINUE_ITERS, lr=CONTINUE_LR)
RUN_DIR = Path(anchor_exp.run_dir)
BASELINE = package(anchor_exp, use_trained={'medium':False,'hard':False},
                   folder_name='anchor_candidate')
print('학습 전 candidate:', BASELINE)
for level, spec in ANCHORS.items():
    print(level, spec)

def run_official(candidate, level, label):
    output = RUN_DIR/level/'anchor_official_eval'/candidate.name/label
    output.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(UPSTREAM/'eval.py'), 'difficulty='+level,
        'obs_mode=state', 'policy=stage_policy:load_policy',
        'checkpoint='+str(candidate/'checkpoints'/level/'model.pt'),
        'eval_config='+str(OFFICIAL), 'max_episode_steps='+str(MAX_STEPS),
        'hydra.run.dir='+str(output)]
    child_env = dict(os.environ)
    child_env['PYTHONPATH'] = str(candidate)+os.pathsep+str(UPSTREAM)+os.pathsep+child_env.get('PYTHONPATH','')
    log = output/'official_eval.log'
    with log.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=UPSTREAM, env=child_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            errors='replace', bufsize=1)
        try:
            for line in process.stdout:
                handle.write(line); handle.flush(); print(line, end='', flush=True)
            code = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                try: process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill(); process.wait()
            process.stdout.close()
    if code:
        raise RuntimeError(f'official eval failed ({level}); {log} 확인')
    videos = sorted((output/'videos').rglob('*.mp4'), key=lambda p:p.stat().st_mtime)
    if videos:
        display(Video(str(videos[-1]), embed=True, width=900))
    return log


GitHub 복원: 0 files
로컬 작업 경로: /content/moveboxes_runs/moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark
실행 노트북 사본: /content/moveboxes_runs/moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark/moveboxes_stages_colab.ipynb
새 런타임에서는 01~03 셀로 결과를 복원하고, 학습·평가는 04~05 셀 준비 후 실행합니다.
GitHub 백업 완료: common (52 files)
단계 ACT: 학습한 완료/복구 판단 + 집기/운반/놓기 행동, 별도 실험
난이도별 검증 앵커 복원: {'easy': '/content/moveboxes_runs/moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark/easy/anchor.pt', 'medium': '/content/moveboxes_runs/moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark/medium/anchor.pt', 'hard': '/content/moveboxes_runs/moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark/hard/anchor.pt'}
학습 전 candidate: /content/moveboxes_runs/moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark/anchor_candidate
easy {'run': 'moveboxes_easy_lab_v1_benchmark', 'checkpoint': 'block_03.pt', 'sha256': '0fbbef92aacc5cbe5642568ec5bd349979934e50f3d03d5120ccd5350720ef19', 'recorded_score': 1.0, 'recorded_episodes'

In [7]:
# 07 · 학습 전 공식 default test · 난이도마다 등록된 앵커 사용
for level in ('easy','medium','hard'):
    run_official(BASELINE, level, 'anchor_default')


/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:42: UserWarning: Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.
  warn("Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.")
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
[eval] git=6048f33217f26ae39009a812f53c81171517f393
----------------------------------------------------------------------
difficulty:
  name: easy
  num_parcels: 2
  fixed_poses: true
randomization:
  parcel_pose:

/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:42: UserWarning: Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.
  warn("Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.")
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
[eval] git=6048f33217f26ae39009a812f53c81171517f393
----------------------------------------------------------------------
difficulty:
  name: medium
  num_parcels: 4
  fixed_poses: false
randomization:
  parcel_po

/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:42: UserWarning: Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.
  warn("Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.")
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
[eval] git=6048f33217f26ae39009a812f53c81171517f393
----------------------------------------------------------------------
difficulty:
  name: hard
  num_parcels: 6
  fixed_poses: false
randomization:
  parcel_pose

## 추가 학습

Easy는 100/100 모델을 그대로 둡니다. Medium과 Hard만 각자의 앵커에서 시작합니다. 셀 재실행 시 같은 난이도의 `latest.pt`와 optimizer/RNG가 복구됩니다. 성능 판단에는 공식 `eval.py` 출력만 사용합니다.


In [11]:
# 09 · GitHub 백업 지연은 경고로 처리 → Medium 1500 step부터 재개 → 공식 test + 영상
from github_store import AssetUploadError

# 학습 subprocess: 422가 나도 다음 checkpoint에서 다시 백업하며 학습은 계속합니다.
store_file = UPSTREAM/'github_store.py'
store_source = store_file.read_text(encoding='utf-8')
old_sync = "    _SYNC_STORES[key].sync(root, key[3], files, refresh=False)"
new_sync = """    try:
        _SYNC_STORES[key].sync(root, key[3], files, refresh=False)
    except AssetUploadError as error:
        print('GitHub 체크포인트 백업 지연; 로컬 학습은 계속합니다:', error, flush=True)"""
if old_sync in store_source:
    store_file.write_text(store_source.replace(old_sync, new_sync), encoding='utf-8')

# 노트북 부모 프로세스: 완료된 계산을 마지막 백업 422 때문에 실패로 바꾸지 않습니다.
_parent_sync_level = anchor_exp.sync_level
_parent_sync_common = anchor_exp.sync_common
def safe_sync_level(level):
    try:
        return _parent_sync_level(level)
    except AssetUploadError as error:
        print('GitHub 난이도 백업 지연; 로컬 결과는 정상입니다:', error)
def safe_sync_common():
    try:
        return _parent_sync_common()
    except AssetUploadError as error:
        print('GitHub 공통 백업 지연; 로컬 결과는 정상입니다:', error)
anchor_exp.sync_level = safe_sync_level
anchor_exp.sync_common = safe_sync_common
print('백업 지연 방어선 적용; latest.pt 1500 step부터 이어갑니다.')

train(anchor_exp, 'medium')
MEDIUM_TRAINED = package(anchor_exp, use_trained={'medium':True,'hard':False},
                         folder_name='medium_trained_candidate')
run_official(MEDIUM_TRAINED, 'medium', 'trained_default')


백업 지연 방어선 적용; latest.pt 1500 step부터 이어갑니다.
실행: /usr/bin/python3 /content/berlin-marso-hackathon/stage_train.py /content/moveboxes_runs/moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark/medium/stage_train_job.json
전체 로그: /content/moveboxes_runs/moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark/medium/train.log
1726/8000 loss=0.0145 22.6 it/s ETA=4.6 min
1973/8000 loss=0.0206 23.6 it/s ETA=4.3 min
GitHub 체크포인트 백업 지연; 로컬 학습은 계속합니다: GitHub asset upload failed (422); local files are retained.
2264/8000 loss=0.0250 18.8 it/s ETA=5.1 min
GitHub 저장 공간 이어쓰기: run-moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark-part-005
2738/8000 loss=0.0060 18.0 it/s ETA=4.9 min
2985/8000 loss=0.0144 18.8 it/s ETA=4.4 min
GitHub 체크포인트 백업 지연; 로컬 학습은 계속합니다: GitHub asset upload failed (422); local files are retained.
3238/8000 loss=0.0287 17.5 it/s ETA=4.5 min
3473/8000 loss=0.0066 18.1 it/s ETA=4.2 min
GitHub 저장 공간 이어쓰기: run-moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark-part-007
3736/8000

PosixPath('/content/moveboxes_runs/moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark/medium/anchor_official_eval/medium_trained_candidate/trained_default/official_eval.log')

In [12]:
# 09 · GitHub 백업 지연은 경고로 처리 → Medium 1500 step부터 재개 → 공식 test + 영상
from github_store import AssetUploadError

# 학습 subprocess: 422가 나도 다음 checkpoint에서 다시 백업하며 학습은 계속합니다.
store_file = UPSTREAM/'github_store.py'
store_source = store_file.read_text(encoding='utf-8')
old_sync = "    _SYNC_STORES[key].sync(root, key[3], files, refresh=False)"
new_sync = """    try:
        _SYNC_STORES[key].sync(root, key[3], files, refresh=False)
    except AssetUploadError as error:
        print('GitHub 체크포인트 백업 지연; 로컬 학습은 계속합니다:', error, flush=True)"""
if old_sync in store_source:
    store_file.write_text(store_source.replace(old_sync, new_sync), encoding='utf-8')

# 노트북 부모 프로세스: 완료된 계산을 마지막 백업 422 때문에 실패로 바꾸지 않습니다.
_parent_sync_level = anchor_exp.sync_level
_parent_sync_common = anchor_exp.sync_common
def safe_sync_level(level):
    try:
        return _parent_sync_level(level)
    except AssetUploadError as error:
        print('GitHub 난이도 백업 지연; 로컬 결과는 정상입니다:', error)
def safe_sync_common():
    try:
        return _parent_sync_common()
    except AssetUploadError as error:
        print('GitHub 공통 백업 지연; 로컬 결과는 정상입니다:', error)
anchor_exp.sync_level = safe_sync_level
anchor_exp.sync_common = safe_sync_common
print('백업 지연 방어선 적용; latest.pt 1500 step부터 이어갑니다.')

train(anchor_exp, 'medium')
MEDIUM_TRAINED = package(anchor_exp, use_trained={'medium':True,'hard':False},
                         folder_name='medium_trained_candidate')
run_official(MEDIUM_TRAINED, 'medium', 'trained_default')


백업 지연 방어선 적용; latest.pt 1500 step부터 이어갑니다.
[medium] 완료된 단계 ACT 모델 재사용. 다음 테스트/평가 셀을 실행하세요.
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:42: UserWarning: Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.
  warn("Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.")
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
[eval] git=6048f33217f26ae39009a812f53c81171517f393
----------------------------------------------------------------------


PosixPath('/content/moveboxes_runs/moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark/medium/anchor_official_eval/medium_trained_candidate/trained_default/official_eval.log')

In [13]:
# 10 · Medium 앵커를 다시 공식 test하여 같은 실행의 직접 비교 자료 생성
run_official(BASELINE, 'medium', 'anchor_repeat')


/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:42: UserWarning: Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.
  warn("Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.")
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
[eval] git=6048f33217f26ae39009a812f53c81171517f393
----------------------------------------------------------------------
difficulty:
  name: medium
  num_parcels: 4
  fixed_poses: false
randomization:
  parcel_po

PosixPath('/content/moveboxes_runs/moveboxes_stage_anchor_v1_anchor_continue_v1_benchmark/medium/anchor_official_eval/anchor_candidate/anchor_repeat/official_eval.log')